# CliniFlow: vLLM rehearsal on Colab (T4)

This notebook is a thin wrapper. It runs the same `serving/` files that Docker and the AWS instance use:
`start_server.sh` starts the server, `client.py` sends a request, and `load_test.py` measures speed.

Before running: Runtime → Change runtime type → **T4 GPU**. Add a Colab secret named `HF_TOKEN` (your Hugging Face token).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# vLLM is large. Install once per session.
!pip install -q vllm

In [ ]:
# Pull the serving files and val set from Drive (no GitHub needed).
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('/content/CliniFlow/serving', exist_ok=True)
os.makedirs('/content/CliniFlow/training/colab', exist_ok=True)
shutil.copytree('/content/drive/MyDrive/cliniflow/serving', '/content/CliniFlow/serving', dirs_exist_ok=True)
shutil.copy('/content/drive/MyDrive/cliniflow/training/colab/val.jsonl', '/content/CliniFlow/training/colab/val.jsonl')
%cd /content/CliniFlow

In [ ]:
# Download the trained adapter from the private HF repo.
import os
from google.colab import userdata
from huggingface_hub import snapshot_download

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
snapshot_download(repo_id="paulerpaul/CliniFlow", local_dir="/content/adapter",
                  allow_patterns=["adapter_*", "tokenizer*", "chat_template.jinja"])
print(os.listdir("/content/adapter"))

`system_prompt.txt` isn't in the HF repo. Copy it from the S3 registry or Google Drive (`adapters/config_A/system_prompt.txt`) into `/content/adapter/` before the next step. The client refuses to run without it.

In [ ]:
# Start the server in the background, same script as Docker.
import subprocess, time, urllib.request

env = dict(os.environ, ADAPTER_DIR="/content/adapter", BASE_MODEL="unsloth/Llama-3.2-3B-Instruct")
server = subprocess.Popen(["bash", "serving/start_server.sh"], env=env,
                          stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)

# Wait until the server answers. Model loading takes a few minutes on a T4.
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("server ready")
        break
    except Exception:
        time.sleep(5)
else:
    print(open("/content/vllm.log").read()[-3000:])
    raise RuntimeError("server did not start, see log above")

In [ ]:
import json
with open("training/colab/val.jsonl") as f:
    example = json.loads(f.readline())
with open("/content/transcript.txt", "w") as f:
    f.write(example["input"])

!python serving/client.py --transcript-file /content/transcript.txt

In [ ]:
# Small load test: 20 requests, 5 at a time.
!python serving/load_test.py --transcripts-file training/colab/val.jsonl --n 20 --concurrency 5

In [ ]:
# Stop the server when finished. Colab bills nothing, but this frees the GPU.
server.terminate()